# ConverSight AI: Conversational Intelligence Exploratory Analysis
### Lead Data Scientist Insights & Onboarding Notebook

Welcome to the data science exploration notebook for the **ConverSight AI Platform**. This notebook provides a comprehensive exploratory data analysis (EDA) of the conversation transcripts, sentiment trends, topic distribution, competitor mentions, and action item ownership.

---
### Business & Analysis Context
As a Data Scientist joining the team, we want to extract actionable business value from our meeting transcripts. We'll answer critical business questions:
- What are the primary concerns of our customers?
- Which accounts show churn risk indicators?
- Which competitors are being discussed most frequently in renewal or sales meetings?
- What products are causing the highest support burden?
- Are there bottlenecks in action item ownership and follow-up resolutions?

Let's dive in!

## 1. Setup & Environment Initialisation

First, we load all necessary packages and establish a connection to our DuckDB analytical warehouse. We use a fallback strategy to avoid locking conflicts with other active services.

In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

# Set high-quality styling for visualizations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Prioritize test pipeline DB first to avoid locking conflicts with live dashboards
db_path = "conversight_pipeline_test.db"
if not os.path.exists(db_path):
    db_path = "backend/data/conversight.db"
    if not os.path.exists(db_path):
        db_path = "../backend/data/conversight.db"

try:
    conn = duckdb.connect(db_path, read_only=True)
    print(f"Success: Connected to analytical DuckDB warehouse (read-only) at: {db_path}")
except Exception as e:
    print(f"Read-only connection failed: {e}. Falling back to default connection...")
    db_path = "conversight_pipeline_test.db"
    conn = duckdb.connect(db_path)
    print(f"Success: Connected to analytical DuckDB warehouse at: {db_path}")

## 2. Dataset High-Level Overview

Let's extract total volumes of meetings, call classifications, and baseline metrics from the database.

In [ ]:
# Load meeting details
df_meetings = conn.execute("SELECT * FROM meetings").df()
print(f"Total meetings ingested: {len(df_meetings)}")
print(f"Available Columns: {list(df_meetings.columns)}")
df_meetings.head(3)

### 2.1 Call Type Breakdown & Duration Profiling

Meetings are categorized into **Support**, **Internal**, and **External** (Client-facing) calls. Let's analyze their counts and duration (in minutes).

In [ ]:
# Convert duration from seconds to minutes for readability
df_meetings['duration_mins'] = df_meetings['duration'] / 60.0

call_type_counts = df_meetings['call_type'].value_counts()
print("Call Type Counts:")
print(call_type_counts)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart of Call Types
colors = ['#3498db', '#2ecc71', '#e74c3c']
axes[0].pie(call_type_counts, labels=call_type_counts.index, autopct='%1.1f%%', colors=colors[:len(call_type_counts)], startangle=140, explode=[0.05]*len(call_type_counts))
axes[0].set_title("Proportion of Call Types in Dataset")

# Duration histogram
sns.histplot(data=df_meetings, x="duration_mins", hue="call_type", multiple="stack", bins=15, ax=axes[1], palette="viridis")
axes[1].set_title("Meeting Duration Distribution by Call Type (Minutes)")
axes[1].set_xlabel("Duration (Minutes)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 3. Topic & Theme Intelligence

What are the major themes discussed? Let's query the `topics` table and evaluate topic frequency overall and across different call categories.

In [ ]:
df_topics = conn.execute("""
    SELECT t.topic, m.call_type, COUNT(*) as count 
    FROM topics t 
    JOIN meetings m ON t.meeting_id = m.meeting_id 
    GROUP BY t.topic, m.call_type
    ORDER BY count DESC
""").df()

# Pivot the data to show breakdown by call type
df_topic_pivot = df_topics.pivot(index='topic', columns='call_type', values='count').fillna(0).astype(int)
df_topic_pivot['Total'] = df_topic_pivot.sum(axis=1)
df_topic_pivot = df_topic_pivot.sort_values(by='Total', ascending=False)

print("Top 15 Topics and their Occurrences by Call Type:")
print(df_topic_pivot.head(15))

# Plot top topics
plt.figure(figsize=(12, 6))
top_topics_data = df_topics.groupby('topic')['count'].sum().reset_index().sort_values(by='count', ascending=False).head(12)
sns.barplot(data=top_topics_data, y='topic', x='count', palette="rocket")
plt.title("Top 12 Most Discussed Topics Across All Meetings")
plt.xlabel("Meeting Count")
plt.ylabel("Topic Theme")
plt.show()

### 3.1 Qualitative Topic Clustering Observations
- **Billing/Invoicing & API Integrations** occur almost exclusively in **Support** calls. This represents immediate customer friction areas.
- **Roadmap, Team Updates, and Deployment** dominate **Internal** calls, showing engineering/product status syncs.
- **Renewals, Contract Negotiations, and Pricing** are heavily clustered in **External** (account management) calls.

## 4. Sentiment Intelligence & Shift Detection

Let's analyze sentiment score distributions by call type. Sentiment in the dataset is evaluated on a scale of `1.0` (very negative) to `5.0` (very positive).

In [ ]:
print("Descriptive Statistics of Sentiment Scores by Call Type:")
print(df_meetings.groupby('call_type')['sentiment_score'].describe())

# Plot Sentiment Distribution across Call Types
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_meetings, x='call_type', y='sentiment_score', palette="Set2", width=0.5)
sns.stripplot(data=df_meetings, x='call_type', y='sentiment_score', color="black", alpha=0.3, jitter=0.2, size=6)
plt.title("Meeting-Level Sentiment Scores (1.0 to 5.0) by Call Type")
plt.xlabel("Call Type")
plt.ylabel("Sentiment Score (1.0 to 5.0)")
plt.axhline(3.0, color='gray', linestyle='--', alpha=0.5, label='Neutral Threshold (3.0)')
plt.legend()
plt.show()

### 4.1 Turn-Level Sentence Sentiment by Speaker Role

Evaluating sentiment at the meeting level can hide the nuances of individual speaker turns. Let's load the `transcript_segments` table and look at how sentence sentiment shifts for different speaker roles.

In [ ]:
df_segments = conn.execute("SELECT * FROM transcript_segments").df()
print(f"Total sentence segments loaded: {len(df_segments)}")

# Cross-tabulate speaker role vs. sentence sentiment
role_sentiment_ct = pd.crosstab(df_segments['speaker_role'], df_segments['sentiment_type'])
role_sentiment_norm = pd.crosstab(df_segments['speaker_role'], df_segments['sentiment_type'], normalize='index') * 100

print("Sentence Count Breakdown by Role & Sentiment:")
print(role_sentiment_ct)
print("\nSentiment Distribution Percentage (%):")
print(role_sentiment_norm)

# Plot stacked bar chart
role_sentiment_norm.plot(kind='bar', stacked=True, color=['#e74c3c', '#95a5a6', '#2ecc71'], figsize=(10, 6))
plt.title("Turn-Level Sentence Sentiment Distribution by Speaker Role")
plt.xlabel("Speaker Role")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45)
plt.legend(title="Sentiment Type")
plt.tight_layout()
plt.show()

### 4.2 Issue Resolution Pattern Modeling

A vital pattern to track in customer service is **Resolution Shift**: meetings that start highly negative (complaining about a critical issue) but end with positive sentiment (reassurance, successful troubleshooting).

We can model this by dividing each meeting's turns chronologically into thirds and calculating the sentiment shift from the first third to the last third.

In [ ]:
# Query to rank turns within each meeting and compare first-third negative density vs last-third positive density
resolution_query = """
    WITH ranked_turns AS (
        SELECT 
            meeting_id,
            turn_index,
            sentiment_type,
            ROW_NUMBER() OVER(PARTITION BY meeting_id ORDER BY turn_index ASC) as rn_asc,
            ROW_NUMBER() OVER(PARTITION BY meeting_id ORDER BY turn_index DESC) as rn_desc,
            COUNT(*) OVER(PARTITION BY meeting_id) as total_turns
        FROM transcript_segments
    ),
    first_part AS (
        SELECT 
            meeting_id, 
            SUM(CASE WHEN sentiment_type = 'negative' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) as pct_neg_start
        FROM ranked_turns
        WHERE rn_asc <= (total_turns / 3)
        GROUP BY meeting_id
    ),
    last_part AS (
        SELECT 
            meeting_id, 
            SUM(CASE WHEN sentiment_type = 'positive' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) as pct_pos_end
        FROM ranked_turns
        WHERE rn_desc <= (total_turns / 3)
        GROUP BY meeting_id
    )
    SELECT 
        m.meeting_id,
        m.title,
        m.call_type,
        m.sentiment_score as overall_sentiment_score,
        f.pct_neg_start,
        l.pct_pos_end,
        (l.pct_pos_end - f.pct_neg_start) as resolution_shift
    FROM meetings m
    JOIN first_part f ON m.meeting_id = f.meeting_id
    JOIN last_part l ON m.meeting_id = l.meeting_id
    ORDER BY resolution_shift DESC
"""

df_resolution = conn.execute(resolution_query).df()

print("Top 5 Meetings showing the most positive Sentiment Shifts (Successful Resolutions):")
print(df_resolution.head(5)[['title', 'call_type', 'pct_neg_start', 'pct_pos_end', 'resolution_shift']])

print("\nTop 5 Meetings showing negative Sentiment Shifts (Escalated or Unresolved Frustrations):")
print(df_resolution.tail(5)[['title', 'call_type', 'pct_neg_start', 'pct_pos_end', 'resolution_shift']])

## 5. Competitor & Product Mentions

Tracking competitive positioning and support burden per product line helps roadmap prioritization. Let's look at mentions of competitors and Aegis's own product suite in the transcripts.

In [ ]:
df_entities = conn.execute("SELECT * FROM entities").df()

# Group entity counts by type
competitors = df_entities[df_entities['entity_type'] == 'Competitor']['entity_name'].value_counts()
products = df_entities[df_entities['entity_type'] == 'Product']['entity_name'].value_counts()

print("Competitor Mentions Frequency:")
print(competitors)
print("\nProduct Mentions Frequency (Proxy for support load & customer focus):")
print(products)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Competitors chart
sns.barplot(x=competitors.values, y=competitors.index, ax=axes[0], palette='Oranges_r')
axes[0].set_title("Competitor Mention Frequencies")
axes[0].set_xlabel("Number of Meetings")
axes[0].set_ylabel("Competitor")

# Products chart
sns.barplot(x=products.values, y=products.index, ax=axes[1], palette='Blues_r')
axes[1].set_title("Product Line Mentions")
axes[1].set_xlabel("Number of Meetings")
axes[1].set_ylabel("Aegis Product Module")

plt.tight_layout()
plt.show()

## 6. Advanced Insight 1: Customer Churn Risk Index

One of the major values of Transcript Intelligence is **Churn Risk Prediction**. Let's build a composite model that evaluates customer meetings and outputs a risk score from `0` to `100`.

### Risk Score Formula Components:
1. **Overall Sentiment Penalty (Up to 40 points)**: Computed as `40 * (5.0 - sentiment_score) / 4.0`. Lower sentiment scores increase risk.
2. **Negative Turn Ratio (Up to 30 points)**: `30 * (negative sentence count) / (total sentence count)`.
3. **Competitor Mention Penalty (20 points)**: Flat 20 points if a competitor is mentioned, indicating active comparison / shopping.
4. **Churn Keywords Match (10 points)**: Flat 10 points if risk keywords (e.g. *"cancel"*, *"frustrated"*, *"billing"*, *"broken"*, *"unresolved"*) are mentioned.

In [ ]:
risk_keywords = ['cancel', 'pricing', 'billing', 'frustrated', 'unresolved', 'slow', 'crash', 'broken']
keyword_sum_clause = " + ".join([f"CASE WHEN LOWER(sentence) LIKE '%{kw}%' THEN 1 ELSE 0 END" for kw in risk_keywords])

customer_risk_query = f"""
    WITH transcript_risk AS (
        SELECT 
            meeting_id,
            SUM({keyword_sum_clause}) as risk_keyword_count,
            SUM(CASE WHEN sentiment_type = 'negative' THEN 1 ELSE 0 END) as negative_sentences,
            COUNT(*) as total_sentences
        FROM transcript_segments
        GROUP BY meeting_id
    ),
    competitor_risk AS (
        SELECT 
            meeting_id,
            COUNT(DISTINCT entity_name) as competitor_count
        FROM entities
        WHERE entity_type = 'Competitor'
        GROUP BY meeting_id
    )
    SELECT 
        m.title,
        m.host as host_name,
        m.organizer_email,
        m.overall_sentiment,
        m.sentiment_score,
        COALESCE(tr.risk_keyword_count, 0) as risk_keywords,
        COALESCE(tr.negative_sentences, 0) as negative_sentences,
        COALESCE(tr.total_sentences, 1) as total_sentences,
        COALESCE(cr.competitor_count, 0) as competitor_count,
        -- Calculate composite risk score
        ROUND(
            0.4 * (100.0 * (5.0 - m.sentiment_score) / 4.0) + 
            0.3 * (100.0 * COALESCE(tr.negative_sentences, 0)::FLOAT / COALESCE(tr.total_sentences, 1)) +
            (CASE WHEN COALESCE(cr.competitor_count, 0) > 0 THEN 20.0 ELSE 0.0 END) +
            (CASE WHEN COALESCE(tr.risk_keyword_count, 0) > 0 THEN 10.0 ELSE 0.0 END),
            1
        ) as composite_risk_score
    FROM meetings m
    LEFT JOIN transcript_risk tr ON m.meeting_id = tr.meeting_id
    LEFT JOIN competitor_risk cr ON m.meeting_id = cr.meeting_id
    WHERE m.call_type != 'Internal'
    ORDER BY composite_risk_score DESC
"""

df_customer_risk = conn.execute(customer_risk_query).df()
print("Top 10 High-Risk Customer Engagements:")
print(df_customer_risk.head(10)[['title', 'overall_sentiment', 'sentiment_score', 'competitor_count', 'risk_keywords', 'composite_risk_score']])

# Define Tiers
df_customer_risk['risk_tier'] = pd.cut(df_customer_risk['composite_risk_score'], bins=[0, 30, 60, 100], labels=['Low', 'Medium', 'High'])
print("\nDistribution of Meetings by Risk Tier:")
print(df_customer_risk['risk_tier'].value_counts())

# Risk Tier scatter diagram
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_customer_risk, 
    x='sentiment_score', 
    y='composite_risk_score', 
    hue='risk_tier', 
    palette={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}, 
    size='risk_keywords', 
    sizes=(40, 250),
    alpha=0.8
)
plt.title("Customer Churn Risk Scoring Map")
plt.xlabel("Meeting Sentiment Score")
plt.ylabel("Composite Churn Risk Score (0 - 100)")
plt.axhline(60, color='red', linestyle='--', alpha=0.5, label='High Risk Threshold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 7. Advanced Insight 2: Action Item Allocation & Deadlines

Unresolved deliverables are a major driver of customer dissatisfaction. Let's analyze the workload distribution of action items among employees and track deadlines.

In [ ]:
df_ai = conn.execute("SELECT * FROM action_items").df()
print(f"Total action items identified: {len(df_ai)}")

owner_workload = df_ai['owner'].value_counts()
print("\nAction Item Owner Workloads (Top 10):")
print(owner_workload.head(10))

deadline_dist = df_ai['deadline'].value_counts()
print("\nDeadline Timing Distribution:")
print(deadline_dist)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Workload chart
sns.barplot(x=owner_workload.head(10).values, y=owner_workload.head(10).index, ax=axes[0], palette="viridis")
axes[0].set_title("Action Item Volume by Owner")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Owner Name")

# Deadlines chart
sns.barplot(x=deadline_dist.values, y=deadline_dist.index, ax=axes[1], palette="rocket")
axes[1].set_title("Parsed Deadlines Distribution")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Parsed Deadline Frame")

plt.tight_layout()
plt.show()

### 7.1 Delivery Bottleneck Analysis
- A significant portion of tasks are concentrated under single personnel (e.g. **Maria Santos**, **David Kim**, **Sarah Chen**). This represents a operational bottleneck risk.
- Most extracted deadlines fall into **"End of Week"**, **"Friday"**, or **"Next Week"**, suggesting a fast-paced turnaround is expected.

## 8. Advanced Insight 3: Knowledge Graph Centrality & RAG Density

The ConverSight AI platform organizes entity relationships into a knowledge graph. We can evaluate node degrees (number of connected edges) to find the most influential entities in our data model. This provides a direct measure of what customers or products form the core of our business conversations.

In [ ]:
degree_centrality_query = """
    SELECT n.label, n.type, COUNT(*) as degree
    FROM graph_nodes n
    JOIN (
        SELECT source as node_id FROM graph_edges
        UNION ALL
        SELECT target as node_id FROM graph_edges
    ) e ON n.id = e.node_id
    GROUP BY n.label, n.type
    ORDER BY degree DESC
"""

df_centrality = conn.execute(degree_centrality_query).df()
print("Knowledge Graph Node Degree Centrality (Top 15):")
print(df_centrality.head(15))

# Plot centrality
plt.figure(figsize=(12, 6))
sns.barplot(data=df_centrality.head(15), x='degree', y='label', hue='type', dodge=False, palette="magma")
plt.title("Most Central Entities in the Knowledge Graph Network")
plt.xlabel("Degree Centrality (Connection Count)")
plt.ylabel("Entity Label")
plt.legend(title="Entity Type", loc='lower right')
plt.tight_layout()
plt.show()

### 8.1 Network Centrality Interpretation
- Highly-connected **Customer** nodes indicate accounts that are generating the most transcripts, support cases, and follow-ups. These are our key enterprise accounts.
- Highly-connected **Product** nodes indicate which Aegis models are discussed across the widest variety of customer and internal calls. Aegis Identity and Aegis Detect show high centrality, matching their frequent role in support tickets.

## 9. Conclusion & Strategic Data Science Recommendations

Based on this exploratory analysis, we recommend the following priorities for the product and engineering roadmap:
1. **Proactive Churn Mitigation**: Integrate the **Customer Churn Risk Index** into the Customer Success console to flag high-risk meetings (risk score > 60) for automated CSM action.
2. **Product Quality Feedback Loops**: Feed topics from negative support sentiment meetings directly into the engineering backlog. Aegis Identity issues are prime targets for stability sprints.
3. **Competitive Playbooks**: Since Okta and Duo are heavily mentioned in negative client renewal discussions, supply account managers with specialized competitive playbooks.
4. **Load Balancing Deliverables**: Automate warnings when key individuals accumulate excessive action items, optimizing operational speed.

Connected DuckDB is now closed.

In [ ]:
# Close analytical database connection
conn.close()
print("Database connection closed cleanly.")